In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
device = "cuda"
path = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
model = AutoModelForSequenceClassification.from_pretrained(path, device_map=device, 
                               trust_remote_code=True, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", use_fast=True)
# We load a random sample from the validation set of the HelpSteer dataset
prompt = 'What are some synonyms for the word "beautiful"?'
response = "Nicely, Beautifully, Handsome, Stunning, Wonderful, Gorgeous, Pretty, Stunning, Elegant"
messages = [{"role": "user", "content": prompt},
           {"role": "assistant", "content": response}]
input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)
with torch.no_grad():
   output = model(input_ids)
   # Multi-objective rewards for the response
   multi_obj_rewards = output.rewards.cpu().float() 
   # The gating layer's output is conditioned on the prompt
   gating_output = output.gating_output.cpu().float()
   # The preference score for the response, aggregated from the 
   # multi-objective rewards with the gating layer
   preference_score = output.score.cpu().float()  
# We apply a transformation matrix to the multi-objective rewards
# before multiplying with the gating layer's output. This mainly aims
# at reducing the verbosity bias of the original reward objectives
obj_transform = model.reward_transform_matrix.data.cpu().float()
# The final coefficients assigned to each reward objective
multi_obj_coeffs = gating_output @ obj_transform.T
# The preference score is the linear combination of the multi-objective rewards with
# the multi-objective coefficients, which can be verified by the following assertion
assert torch.isclose(torch.sum(multi_obj_rewards * multi_obj_coeffs, dim=1), preference_score, atol=1e-3) 
# Find the top-K reward objectives with coefficients of the highest magnitude
K = 3
top_obj_dims = torch.argsort(torch.abs(multi_obj_coeffs), dim=1, descending=True,)[:, :K]
top_obj_coeffs = torch.gather(multi_obj_coeffs, dim=1, index=top_obj_dims)

# The attributes of the 19 reward objectives
attributes = ['helpsteer-helpfulness','helpsteer-correctness','helpsteer-coherence',
   'helpsteer-complexity','helpsteer-verbosity','ultrafeedback-overall_score',
   'ultrafeedback-instruction_following', 'ultrafeedback-truthfulness',
   'ultrafeedback-honesty','ultrafeedback-helpfulness','beavertails-is_safe',
   'prometheus-score','argilla-overall_quality','argilla-judge_lm','code-complexity',
   'code-style','code-explanation','code-instruction-following','code-readability']

example_index = 0
for i in range(K):
   attribute = attributes[top_obj_dims[example_index, i].item()]
   coeff = top_obj_coeffs[example_index, i].item()
   print(f"{attribute}: {round(coeff,5)}")
# code-complexity: 0.19922
# helpsteer-verbosity: -0.10864
# ultrafeedback-instruction_following: 0.07861

# The actual rewards of this example from the HelpSteer dataset
# are [3,3,4,2,2] for the five helpsteer objectives: 
# helpfulness, correctness, coherence, complexity, verbosity
# We can linearly transform our predicted rewards to the 
# original reward space to compare with the ground truth
helpsteer_rewards_pred = multi_obj_rewards[0, :5] * 5 - 0.5
print(helpsteer_rewards_pred)
# [2.78125   2.859375  3.484375  1.3847656 1.296875 ]

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


code-complexity: 0.20215
helpsteer-verbosity: -0.11231
ultrafeedback-instruction_following: 0.08203
tensor([3.5039, 3.2305, 3.4844, 0.7354, 0.7598])


In [3]:
multi_obj_coeffs

tensor([[ 6.7139e-03,  1.3388e-08,  1.8533e-07,  3.5667e-04, -1.1231e-01,
          1.2994e-05,  8.2031e-02,  5.5879e-09,  3.2959e-02,  5.5879e-09,
          4.2021e-06,  3.4869e-06,  1.6308e-04,  9.5844e-05,  2.0215e-01,
          1.8533e-07,  2.0027e-05,  5.8746e-04,  2.2119e-08]])

In [2]:
multi_obj_rewards

tensor([[0.8008, 0.7461, 0.7969, 0.2471, 0.2520, 0.6328, 0.5469, 0.6836, 0.6719,
         0.4453, 1.1641, 0.4160, 0.8203, 0.6172, 0.5430, 0.4141, 0.3770, 0.3301,
         0.3926]])

In [4]:
reward_breakdown = {}
reward_breakdown_coeffs = {}
for index, elem in enumerate(attributes):
    reward_breakdown[elem] = multi_obj_rewards[0, index].item()
    reward_breakdown_coeffs[elem] = multi_obj_coeffs[0, index].item()

In [6]:
reward_breakdown_coeffs

{'helpsteer-helpfulness': 0.0067138671875,
 'helpsteer-correctness': 1.3387762010097504e-08,
 'helpsteer-coherence': 1.8533319234848022e-07,
 'helpsteer-complexity': 0.0003566741943359375,
 'helpsteer-verbosity': -0.1123061552643776,
 'ultrafeedback-overall_score': 1.2993812561035156e-05,
 'ultrafeedback-instruction_following': 0.08203125,
 'ultrafeedback-truthfulness': 5.587935447692871e-09,
 'ultrafeedback-honesty': 0.032958984375,
 'ultrafeedback-helpfulness': 5.587935447692871e-09,
 'beavertails-is_safe': 4.202127456665039e-06,
 'prometheus-score': 3.4868717193603516e-06,
 'argilla-overall_quality': 0.00016307830810546875,
 'argilla-judge_lm': 9.584426879882812e-05,
 'code-complexity': 0.2021484375,
 'code-style': 1.8533319234848022e-07,
 'code-explanation': 2.002716064453125e-05,
 'code-instruction-following': 0.00058746337890625,
 'code-readability': 2.2118911147117615e-08}

In [37]:
tokenizer.decode(input_ids[0]) # with instruct tokenizer
# tensor([3.5039, 3.2305, 3.4844, 0.7354, 0.7598]) -> preference score = 0.1543

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat are some synonyms for the word "beautiful"?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nNicely, Beautifully, Handsome, Stunning, Wonderful, Gorgeous, Pretty, Stunning, Elegant<|eot_id|>'

In [3]:
tokenizer.decode(input_ids[0]) # with ArmoRM tokenizer
# tensor([2.7617, 2.8203, 3.4844, 1.3750, 1.2871]) -> preference score = 0.1099

'<|start_header_id|>user<|end_header_id|>\n\nWhat are some synonyms for the word "beautiful"?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nNicely, Beautifully, Handsome, Stunning, Wonderful, Gorgeous, Pretty, Stunning, Elegant<|eot_id|>'

In [6]:
for attribute, reward, coeff in zip(attributes, multi_obj_rewards[0], multi_obj_coeffs[0]):
    print(attribute, round(reward.item(), 5), round(coeff.item(), 5))

helpsteer-helpfulness 0.65234 0.00641
helpsteer-correctness 0.66406 0.0
helpsteer-coherence 0.79688 0.0
helpsteer-complexity 0.375 0.00011
helpsteer-verbosity 0.35742 -0.10856
ultrafeedback-overall_score 0.58984 1e-05
ultrafeedback-instruction_following 0.57031 0.07861
ultrafeedback-truthfulness 0.69531 0.0
ultrafeedback-honesty 0.64062 0.03015
ultrafeedback-helpfulness 0.46094 0.0
beavertails-is_safe 0.92969 0.0
prometheus-score 0.52734 0.0
argilla-overall_quality 0.53516 9e-05
argilla-judge_lm 0.53906 5e-05
code-complexity 0.40234 0.19922
code-style 0.37891 0.0
code-explanation 0.38086 1e-05
code-instruction-following 0.42188 0.00039
code-readability 0.44922 0.0


In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
instruct_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    padding_side="left",
    trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    padding_side="left",
    trust_remote_code=True,
)
armoRM_tokenizer = AutoTokenizer.from_pretrained("RLHFlow/ArmoRM-Llama3-8B-v0.1", use_fast=True)

In [5]:
base_tokenizer.eos_token, instruct_tokenizer.eos_token, armoRM_tokenizer.eos_token

('<|end_of_text|>', '<|eot_id|>', '<|end_of_text|>')

In [32]:
messages = [{"role": "user", "content": "What's 1+1?"}]
armoRM_tokenizer.decode(armoRM_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt = True)[0])

"<|start_header_id|>user<|end_header_id|>\n\nWhat's 1+1?<|eot_id|>"

In [12]:
import pandas as pd

def display(tokenizer):
    tokens = list(tokenizer.added_tokens_decoder.items())[:11]
    return pd.DataFrame(tokens, columns=['Token ID', 'Token'])
display(base_tokenizer)


,Token ID,Token
0,128000,<|begin_of_text|>
1,128001,<|end_of_text|>
2,128002,<|reserved_special_token_0|>
3,128003,<|reserved_special_token_1|>
4,128004,<|finetune_right_pad_id|>
5,128005,<|reserved_special_token_2|>
6,128006,<|start_header_id|>
7,128007,<|end_header_id|>
8,128008,<|eom_id|>
9,128009,<|eot_id|>


In [13]:
display(instruct_tokenizer)

,Token ID,Token
0,128000,<|begin_of_text|>
1,128001,<|end_of_text|>
2,128002,<|reserved_special_token_0|>
3,128003,<|reserved_special_token_1|>
4,128004,<|finetune_right_pad_id|>
5,128005,<|reserved_special_token_2|>
6,128006,<|start_header_id|>
7,128007,<|end_header_id|>
8,128008,<|eom_id|>
9,128009,<|eot_id|>


In [14]:
display(armoRM_tokenizer)

,Token ID,Token
0,128000,<|begin_of_text|>
1,128001,<|end_of_text|>
2,128002,<|reserved_special_token_0|>
3,128003,<|reserved_special_token_1|>
4,128004,<|reserved_special_token_2|>
5,128005,<|reserved_special_token_3|>
6,128006,<|start_header_id|>
7,128007,<|end_header_id|>
8,128008,<|reserved_special_token_4|>
9,128009,<|eot_id|>
